# Séance 6 : Index textuels
# Prétraitement, index inversé, TF-IDF, BM25, n-grams

**Enseignant :** Jean Delpech

**Cours :** Algorithmie et développement dans l'ingénierie des données

**Classe :** M1 Data

**Année scolaire :** 2025/2026

**Dernière mise à jour :** juin 2026

## Objectifs

- Comprendre pourquoi la recherche textuelle naïve O(n·L) ne passe pas à l'échelle
- Maîtriser la structure et la construction d'un index inversé
- Comprendre TF-IDF comme modèle de pertinence et représentation vectorielle
- Comprendre BM25 comme amélioration de TF-IDF, et pourquoi c'est le standard industriel
- Comprendre les n-grams et la similarité de Jaccard pour la recherche approximative

## Plan

| # | Partie |
|---|---|
| 1 | Le problème : recherche naïve et passage à l'échelle |
| 2 | Prétraitement du texte |
| 3 | Index inversé : structure, construction, requêtes booléennes |
| 4 | TF-IDF : modèle vectoriel de pertinence |
| 5 | BM25 : corrections et standard industriel |
| 6 | N-grams et recherche approximative |
| 7 | Pour aller plus loin — lien avec les mini-mémoires |

> **Note pédagogique**  
> Ce notebook est volontairement théorique. Les algorithmes sont présentés sous forme de pseudo-code commenté et de descriptions d'architecture. L'implémentation *from scratch* constitue le travail attendu dans le cadre des mini-mémoires.

> **Contexte dans le module**  
> Cette séance s'appuie sur les tables de hachage (S2), les listes triées (S3) et les algorithmes de parcours (S4). Elle prépare directement la séance 7 sur les bases vectorielles, dont la représentation par embeddings dense est la généralisation moderne du modèle vectoriel TF-IDF.

# PARTIE 1 : Le problème (recherche naïve et passage à l'échelle)

## 1.1 Pourquoi la recherche naïve ne fonctionne pas

Imaginez un moteur de recherche devant retrouver tous les documents contenant le mot `"hachage"` parmi 10 millions d'articles. L'approche la plus simple : parcourir chaque document et chercher le mot dedans. C'est la **recherche naïve**, ou *force brute*.

Sa complexité est **O(n · L)**, où :
- `n` = nombre de documents dans le corpus
- `L` = longueur moyenne d'un document (en caractères)

Sur 10 millions de documents de 1000 mots (≈ 6000 caractères) : **60 milliards d'opérations par requête**. À raison de quelques milliards d'opérations par seconde pour un CPU moderne, cela représente plus d'une minute de calcul pour une seule requête, ce qui est tout bonnnement inacceptable.

La solution à laquelle on pense immédiatement après quelques séances de ce cours d’algorithmie : **pré-indexer** les données. On fait le travail de découpage et d'organisation *une seule fois* lors de l'ingestion des documents, pour rendre chaque requête quasi-instantanée. C'est le principe de l'**index inversé**.

```
SANS index  :  requête → parcourir les N documents → O(N · L) à chaque requête
AVEC index  :  ingestion → construire l'index  O(N · L), une seule fois
               requête → consulter l'index    O(log V) où V = taille du vocabulaire
```

Le coût O(N · L) existe dans les deux cas, mais avec un index il est payé une seule fois, pas à chaque requête.

# PARTIE 2 : Prétraitement du texte

## 2.1 Pourquoi normaliser avant d'indexer ?

Avant de construire l'index, il faut transformer chaque document brut en une liste de **tokens** normalisés. Sans cette étape :
- `"Hachage"`, `"hachage"` et `"hachage,"` seraient trois entrées différentes dans l'index
- Des mots très fréquents (`"le"`, `"de"`, `"et"`) surchargeraient l'index en ne portant aucune information discriminante
- `"algorithme"` et `"algorithmes"` seraient indexés séparément alors qu'ils désignent la même chose

Le prétraitement réduit le vocabulaire, améliore la couverture, et rend l'index plus compact.

## 2.2 Les étapes classiques

| Étape | Description | Exemple |
|---|---|---|
| **Tokenisation** | Découper le texte en unités élémentaires (tokens) | `"Le chat dort"` → `["Le", "chat", "dort"]` |
| **Normalisation** | Minuscules, suppression de la ponctuation | `"Hachage,"` → `"hachage"` |
| **Suppression des stop words** | Retirer les mots trop fréquents et non discriminants | `"le"`, `"de"`, `"et"` → supprimés |
| **Stemming** | Réduire à la racine morphologique (heuristique) | `"algorithmes"` → `"algorithm"` |
| **Lemmatisation** | Réduire à la forme canonique (plus précis, nécessite un lexique) | `"allant"` → `"aller"` |

### Stemming vs lemmatisation

Le **stemming** coupe les suffixes selon des règles fixes (algorithme de Porter, Snowball). C'est rapide mais grossier : il peut produire des formes qui ne sont pas des mots réels (`"general"` → `"gener"`). La **lemmatisation** utilise un dictionnaire lexical pour trouver la forme canonique : plus précise mais plus coûteuse. Pour un moteur à grande échelle, le stemming est souvent suffisant.

### *Stop words* : précaution importante

La liste de *stop words* doit être choisie avec soin selon le domaine. `"not"` (en anglais) est souvent mis en *stop word*, mais supprimer `"not"` dans `"this product is not recommended"` change radicalement le sens d'un avis client. En NLP de précision, on préfère parfois conserver les stop words et laisser des modèles de pertinence gérer leur faible poids naturellement (leur IDF sera proche de 0, comme on le verra en partie 4).

## 2.3 Architecture d'un module de prétraitement

```
Classe Preprocessor
   Attributs
      stop_words   : Set[str]         ← ensemble des mots à supprimer
      n            : int              ← taille des n-grams (pour la partie 6)

   Méthodes
      tokenize(text: str) → List[str]
         Étape 1 : passer en minuscules
         Étape 2 : extraire les séquences alphanumériques (regex)
         Étape 3 : filtrer les stop words et les tokens trop courts
         Retourne la liste de tokens

      normalize(token: str) → str
         Appliquer stemming ou lemmatisation si configuré
         Retourne le token normalisé

      process(text: str) → List[str]
         = tokenize puis normalize chaque token
         Méthode principale appelée par l'index
```

# PARTIE 3 : Index inversé

## 3.1 Structure

Un **index inversé** est la structure de données centrale de tout moteur de recherche. Son principe est simple :

> Pour chaque terme du vocabulaire, stocker la liste de tous les documents qui le contiennent.

```
{ terme → posting list }
```

Une **posting list** est la liste des identifiants de documents contenant le terme. Elle est systématiquement maintenue **triée** par `doc_id` croissant, cette propriété est fondamentale pour les requêtes multi-termes (voir 3.3).

Exemple sur un petit corpus :
```
{
  "hachage"    : [0, 3, 7, 12],
  "algorithme" : [0, 1, 2, 3, 5, 7],
  "collision"  : [3, 12, 45],
  "index"      : [6, 7, 18, 22],
  ...
}
```

La structure de données qui supporte l'index est une **table de hachage** (accès O(1) au terme) dont les valeurs sont des listes triées. Sur disque, on utilise en général un B-tree pour permettre les requêtes par préfixe (voir mémoire sujet 2).

En plus des ̀`doc_ids`, la posting list peut stocker des informations supplémentaires :
- le **nombre d'occurrences** du terme dans chaque document (ce qu’on verra plus en détail dans la présentation de TF-IDF, où on normalisera cette fréquence, ce qu’on ne fait pas encore ici)
- les **positions** des occurrences (pour les requêtes de proximité : `"machine learning"` comme phrase exacte)

```
# Posting list avec fréquence
"hachage" : [(doc_id=0, term_frequency=2), (doc_id=3, term_frequency=5), (doc_id=7, term_frequency=1), ...]

# Posting list avec positions
"hachage" : [(doc_id=0, positions=[14, 237]), (doc_id=3, positions=[2, 8, 91, ...]), ...]
```

## 3.2 Construction

La construction de l'index se fait en **une seule passe** sur le corpus. C'est le travail fait une seule fois à l'ingestion.

```
CONSTRUIRE_INDEX(corpus) :

   index ← HashTable vide

   pour chaque (doc_id, document) dans corpus :
      tokens ← preprocessor.process(document)

      pour chaque token dans tokens :
         si token absent de index :
            index[token] ← liste vide
         si doc_id absent de index[token] :  ← on évite les doublons
            ajouter doc_id à index[token]

   pour chaque token dans index :
      trier index[token]                      ← tri final O(df_t · log df_t)

   retourner index
```

**Complexité :** O(N · L · log L). N documents, L tokens en moyenne, le log vient du tri final des posting lists. En pratique, le tri est marginal car les doc_ids sont déjà majoritairement croissants si on ingère le corpus dans l'ordre.

**Sur un grand corpus** (milliards de documents), la construction se fait par blocs en parallèle, chaque worker construit un index partiel, puis les index partiels sont fusionnés, exactement le paradigme Map/Reduce (séance 5).

## 3.3 Requêtes booléennes et algorithme à deux pointeurs

### Requête AND

Une requête AND (`"hachage" AND "collision"`) retourne les documents contenant **les deux** termes : c'est l'**intersection** de leurs posting lists.

Grâce au tri, l'intersection se fait en **O(|L1| + |L2|)** avec l'**algorithme à deux pointeurs** : on avance simultanément dans les deux listes et on ne retient que les `doc_ids` communs.

```
INTERSECTER(L1, L2) :
   résultat ← liste vide
   i ← 0 , j ← 0

   tant que i < |L1| et j < |L2| :
      si L1[i] == L2[j] :           ← même doc_id dans les deux listes
         ajouter L1[i] à résultat
         i ← i + 1 , j ← j + 1
      sinon si L1[i] < L2[j] :      ← L1 est en retard : avancer i
         i ← i + 1
      sinon :                        ← L2 est en retard : avancer j
         j ← j + 1

   retourner résultat
```

Illustration sur un exemple :
```
L1 = [0, 3, 7, 12]   (posting list de "hachage")
L2 = [3, 5, 12, 45]  (posting list de "collision")

Étape 1 : L1[0]=0  < L2[0]=3  → avancer i
Étape 2 : L1[1]=3 == L2[0]=3  → commun, ajouter 3, avancer i et j
Étape 3 : L1[2]=7  > L2[1]=5  → avancer j
Étape 4 : L1[2]=7  < L2[2]=12 → avancer i
Étape 5 : L1[3]=12 == L2[2]=12 → commun, ajouter 12, avancer i et j
Fin : j a atteint la fin de L2

Résultat : [3, 12]
```

Sans le tri, l'intersection naïve serait O(|L1| · |L2|). **Le tri des posting lists est donc fondamental** pour la performance des requêtes.

### Optimisation : ordre d'intersection

Pour une requête à k termes, il vaut mieux intersecter en commençant par les **termes les moins fréquents** (posting lists les plus courtes). Cela réduit la taille de l'ensemble intermédiaire le plus tôt possible.

>`df` est la *document frequency* du terme considéré, c'est-à-dire le nombre de documents dans le corpus qui contiennent ce terme, autrement dit la longueur de sa *posting list*. Un terme rare comme "hachage" aura un df faible (courte *posting list*), un terme fréquent comme "données" aura un df élevé (longue *posting list*).
>
>Trier par `df` croissant revient donc à commencer l'intersection par les termes les plus rares, dont la *posting list* est la plus courte, ce qui réduit le plus vite la taille de l'ensemble de candidats. On reviendra sur le terme `df` qu’on définit formellement en partie 4 dans la formule IDF.

```
RECHERCHE_AND(index, termes) :
   # Trier les termes par df croissant (posting list la plus courte en premier)
   termes_triés ← trier termes par len(index[t])
   résultat ← index[termes_triés[0]]
   pour chaque terme dans termes_triés[1:] :
      résultat ← INTERSECTER(résultat, index[terme])
      si résultat est vide : retourner []   ← early stopping
   retourner résultat
```

### Requête OR

La requête OR (`"hachage" OR "collision"`) est l'**union** des *posting lists*. Le même algorithme à deux pointeurs fonctionne, en ajoutant les `doc_ids` des deux listes de façon triée et sans doublons, la complexité identique O(|L1| + |L2|).

### Limite de la recherche booléenne

La recherche booléenne répond à la question *"ce document contient-il les termes ?"* mais pas *"dans quelle mesure est-il pertinent ?"*. Tous les résultats ont le même statut. Pour classer les résultats par pertinence, il faut un **score**, c'est là qu'interviendra TF-IDF.

## 3.4 Architecture d'un index inversé

Finalement, on peut définir une classe `InvertedIndex`qui implémente un index inversé. Pour déduire comment on va créer cette classe, on peut partir de quelques questions (un point méthode intéressant à retenir pour l’appliquer quand on doit créer une classe) : 

> on veut construire un index inversé, qu'est-ce qu'il nous faut ?

### Que doit stocker l'index ? (attributs)

* La structure de données centrale est le dictionnaire `index` lui-même : 
> un terme → sa posting list.

Ici un dictionnaire Python suffit : `index : Dict[str, List[int]]`. 

* On veut aussi savoir combien de documents sont dans le corpus, car cette information sera utile plus tard, autant la stocker dès la construction : `doc_count : int`.
* Enfin, on a vu en partie 2 que le prétraitement doit être cohérent entre la construction et les requêtes, donc on embarque donc une instance de préprocesseur à appliquer au corpus et aux texte des requêtes : `preprocessor : Preprocessor`.

> `InvertedIndex` ne sait pas comment tokeniser un texte car ce n'est tout simplement pas son rôle. Il délègue cette responsabilité à une instance de la classe Preprocessor qu'on lui passe à la construction. C'est le principe de composition vu en séance 1 (clean code et ingénierie logicielle) : plutôt que de faire hériter `InvertedIndex` d'un `Preprocessor` ou de dupliquer la logique de tokenisation, on passe un objet `Preprocessor` en paramètre, ce qui permet de changer facilement de stratégie de prétraitement (avec ou sans *stemming*, avec des *stop words* différents) sans toucher au code de l'index.

### Qu’est-ce qu'on veut faire avec et index ? (méthodes)

* La première opération qu’on devra évidemment réaliser est la construction de l'index depuis un corpus : `build(corpus)`.
* Une fois l'index construit, on a besoin de consulter la *posting list* d'un terme : `posting_list(term)`.
* Une opération dérivée qui en découle directement est de pouvoir connaître le nombre de documents qui contiennent un terme (c.-à-d. la longueur de sa *posting list*), qu'on a appelé *document frequency*. On imagine dès à présent que cette méthode sera suffisamment utile pour en faire une méthode à part : `df(term)`.
  > Note : on serait tenté de mettre un décorateur `@property` à cette méthode, hélas on ne peut pas utiliser un tel décorateur sur des méthodes qui prennent des argument (ici `term`)
* Enfin on veut pouvoir répondre à des requêtes, notamment booléennes, ce qui nous donne `search_and(query)` et `search_or(query)` tels que décrites dans la section précédente.

### Classe InvertedIndex

```
Classe InvertedIndex
   Attributs
      index          : Dict[str, List[int]]      ← terme → posting list triée
      doc_count      : int                       ← N, nombre total de documents
      preprocessor   : Preprocessor

   Méthodes
      build(corpus: List[str]) → None
         Parcourt le corpus, tokenise chaque doc, construit les posting lists
         Trie les posting lists en fin de construction
         Complexité : O(N·L·log L)

      posting_list(term: str) → List[int]
         Retourne la posting list d'un terme (liste vide si absent)
         Complexité : O(1) (cf. accès HashTable)

      df(term: str) → int
         Document frequency : longueur de la posting list
         Complexité : O(1)

      search_and(query: str) → List[int]
         Tokenise la requête, intersecte les posting lists dans l'ordre croissant de df
         Retourne les doc_ids des documents contenant tous les termes
         Complexité : O(k · df_min) où k = nombre de termes, df_min = df du terme le plus rare

      search_or(query: str) → List[int]
         Tokenise la requête, fait l'union des posting lists
         Complexité : O(somme des df_t pour t dans la requête)
```

# PARTIE 4 : TF-IDF

## 4.1 Intuition

TF-IDF (**Term Frequency - Inverse Document Frequency**) répond à la question :

> Un terme est-il *caractéristique* de ce document, ou simplement *fréquent* dans tout le corpus ?

Deux observations contradictoires en apparence :
1. Si le mot `"hachage"` apparaît souvent dans un document → ce document parle probablement de hachage (TF ↑)
2. Si le mot `"le"` apparaît dans quasi tous les documents → il ne permet pas de distinguer les documents entre eux (IDF → 0)

TF-IDF combine ces deux mesures pour donner un score de **pertinence locale** (TF) pondéré par la **rareté globale** (IDF).

## 4.2 Les formules

### TF (Term Frequency)

Fréquence relative du terme `t` dans le document `d` :

$$TF(t, d) = \frac{\text{nombre d'occurrences de } t \text{ dans } d}{\text{nombre total de tokens dans } d}$$

On voit qu’on fait intervenir ici une **normalisation** : on divise la *fréquence brute* (telle qu’évoquée dans la partie précédente) par la *longueur* du document considéré.
La normalisation par la longueur du document est importante : sans elle, un document de 10 000 mots qui mentionne `"hachage"` 10 fois aurait le même score brut qu'un document de 100 mots qui le mentionne aussi 10 fois, alors que dans le second, le terme est bien plus central.

### IDF (Inverse Document Frequency)

$$IDF(t) = \log\left(\frac{N}{df_t}\right)$$

Où :
- $N$ = nombre total de documents dans le corpus
- $df_t$ = *document frequency* : nombre de documents contenant le terme `t`

Interprétation :
- Si `t` est présent dans **tous** les documents (`df_t = N`) : `IDF = log(1) = 0` → score nul, terme non discriminant
- Si `t` est présent dans **un seul** document (`df_t = 1`) : `IDF = log(N)` → score maximal, terme très rare
- Entre les deux, l'IDF varie continûment avec la rareté du terme

Le logarithme joue un rôle essentiel : il *compresse* l'échelle. Sans lui, un terme présent dans 10 documents sur 1 million serait 100 fois plus pénalisé qu'un terme présent dans 1000 documents, alors que les deux sont des termes rares. Le log rend la pénalisation plus progressive.

### Score TF-IDF

$$\text{TF-IDF}(t, d) = TF(t, d) \times IDF(t)$$

Pour une requête multi-termes, le score d'un document est la **somme des TF-IDF** de chaque terme de la requête :

$$\text{score}(q, d) = \sum_{t \in q} TF(t, d) \times IDF(t)$$

## 4.3 Le modèle vectoriel et la similarité cosinus

TF-IDF définit une **représentation vectorielle** de chaque document. On associe à chaque terme du vocabulaire une dimension : le document `d` devient un vecteur $\mathbf{d}$ de dimension $|V|$ (taille du vocabulaire), où la composante associée au terme `t` vaut $TF\text{-}IDF(t, d)$.

Ce vecteur est **creux** (*sparse*) : la plupart des composantes valent 0 (la plupart des termes du vocabulaire n'apparaissent pas dans un document donné).

La pertinence d'un document pour une requête se mesure alors par la **similarité cosinus** entre le vecteur document et le vecteur requête :

$$\cos(\mathbf{d}, \mathbf{q}) = \frac{\mathbf{d} \cdot \mathbf{q}}{\|\mathbf{d}\| \cdot \|\mathbf{q}\|}$$

La similarité cosinus mesure l'**angle** entre les deux vecteurs, indépendamment de leur norme. Un document très long qui parle de hachage et un document court qui parle aussi de hachage auront des vecteurs qui pointent dans la même direction, même si leurs normes diffèrent.

```
Vocabulaire simplifié : ["hachage", "collision", "algorithme", "tri", "graphe"]

Doc 0 (parle de hachage et collision) :
   d0 = [0.42, 0.38, 0.12, 0.0, 0.0]    ← vecteur creux

Doc 1 (parle de tri et algorithme) :
   d1 = [0.0, 0.0, 0.21, 0.55, 0.0]

Requête "hachage collision" :
   q  = [0.5,  0.5,  0.0,  0.0, 0.0]

cos(d0, q) est élevé  → Doc 0 pertinent
cos(d1, q) ≈ 0        → Doc 1 non pertinent
```

> **Lien avec la séance 7 :** les embeddings (BERT, sentence-transformers) sont une généralisation de ce principe. Au lieu d'un vecteur creux de dimension |V| (taille du vocabulaire, souvent 50 000+) construit à partir des fréquences de mots, un embedding est un vecteur *dense* de dimension fixe (768 pour BERT) appris par un réseau de neurones. La similarité cosinus reste la mesure de référence dans les deux cas.

## 4.4 Architecture TF-IDF

Comme précédemment, on va se poser des questions pour nous aider à imaginer comment concevoir cette classe.

On n’a pas 50 points de départ : on va partir des formules vues en section 4.2 et on se demandera ce dont on a besoin pour les calculer.

### Que veut-on calculer ? 

La formule TF-IDF nécessite deux ingrédients : TF et IDF (bravo Captain Obvious). 

Ce sera donc deux méthodes de base qui nous les fournirons :
* `tf(term, doc_id)`
* `idf(term)`.

* À partir de là, scorer un document pour une requête est leur produit sommé sur les termes : `score(query, doc_id)`.
* Et finalement, chercher les meilleurs documents pour une requête, c'est `search(query, top_k)`.

> On voit ici un cas d’usage direct d’un heap (tas binaire) : maintenir les k meilleurs scores sans trier l'intégralité des candidats. `heapq.nlargest(top_k, scored, key=...)` en Python fait exactement ça en O(n log k). 

### De quoi avons nous besoin pour calculer TF ?

La formule est `count(t, d) / longueur(d)`. Il faut donc stocker pour chaque document :
* le nombre brut d'occurrences de chaque terme `tf_raw : Dict[int, Dict[str, int]]`
* la longueur de chaque document en tokens `doc_lengths : Dict[int, int]`

Ces deux structures se remplissent en une seule passe sur le corpus lors du build.

### De quoi avons nous besoin pour calculer IDF ? 

La formule est `log(N / df(term))`
* Il nous faut donc `N` le nombre total de documents
* `df(term)` pour chaque terme. Or `df` est déjà disponible via `InvertedIndex` construit précédemment inutile de le recalculer. On devra donc stocker une référence vers cet index : `index : InvertedIndex, et n_docs : int` pour `N`.

### L’index inversé est aussi utile pour search

Ici scorer tous les `N` documents à chaque requête ramènerait à une complexité O(N) par requête, ce qu'on cherche précisément à éviter. 
L'index inversé nous donne gratuitement la liste des documents qui contiennent au moins un terme de la requête, ce qu’on va appeler les candidats. On ne score donc que ces candidats, qui ne réprésentent généralement qu’une fraction infime de `N`, ce qui constitue un gain de temps non négligeable.

### Conception de la classe TFIDF


```
Classe TFIDF
   Attributs
      index          : InvertedIndex             ← pour accéder aux df
      tf_raw         : Dict[int, Dict[str, int]] ← {doc_id → {terme → count}}
      doc_lengths    : Dict[int, int]            ← longueur (en tokens) de chaque doc
      n_docs         : int                       ← N

   Méthodes
      build(corpus) → None
         Pré-calcule tf_raw et doc_lengths en une passe
         (les df sont déjà dans l'index inversé)

      tf(term, doc_id) → float
         = tf_raw[doc_id][term] / doc_lengths[doc_id]

      idf(term) → float
         = log(n_docs / index.df(term))
         = 0 si le terme est absent du corpus

      score(query, doc_id) → float
         = somme des tf(t, doc_id) * idf(t) pour t dans tokenize(query)

      search(query, top_k) → List[Tuple[int, float]]
         Candidats : union des posting lists des termes de la requête
         Pour chaque candidat : calculer score(query, doc_id)
         Retourner les top_k par score décroissant
         (ne pas scorer tous les documents, seulement les candidats de l'index)
```

> **Note sur la complexité du scoring**
> 
> On ne calcule le score que pour les **candidats**. Les documents présents dans au moins une *posting list* des termes de la requête. Sur un grand corpus, c'est généralement une fraction infime des N documents. C'est tout l'intérêt de l'index : il effectue un pré-filtrage massif avant le calcul des scores.

# PARTIE 5 : BM25

## 5.1 Les deux limites de TF-IDF

TF-IDF est élégant mais présente deux problèmes bien identifiés depuis les années 1990 :

**Problème 1 : La saturation du TF n'est pas bornée**

Si le terme `"hachage"` apparaît 100 fois dans un document, TF-IDF lui attribue un score 100× supérieur à un document qui le mentionne une fois dans un paragraphe synthétique et précis. Intuitivement, répéter un mot 100 fois n'indique pas une pertinence 100× supérieure : cela peut même indiquer un contenu de mauvaise qualité (bourrage de mots-clés, *keyword stuffing*). On utilise le mot *saturation* pour désigner le fait que la contribution d'un terme au score devrait plafonner à partir d'un certain nombre d'occurrences. Pour revenir à un exemple, si le mot "hachage" apparaît 3 fois dans un document, on peut raisonnablement conclure que ce document parle de hachage. S'il apparaît 30 fois, il en parle probablement encore plus, mais est-il vraiment 10 fois plus pertinent ? Non, on devrait considérer qu’à partir de 3 apparaitions sur un paragraphe il y a « saturation » et cela n’apporte aucune pertinence d’en parler plus. La contribution d’un terme devrait converger vers un plafond quelque soit l’augmentation de sa fréquence après un certain point. On a chercher à créer une formule qui présente ce comportement.

**Problème 2 : Les documents longs sont systématiquement favorisés**

La normalisation par la longueur dans le TF ($\frac{\text{count}}{|d|}$) corrige partiellement le biais précédent, mais elle fait l’hypothèse d’une relation linéaire entre la longueur et la fréquence qui ne tient pas toujours. Un long document généraliste qui mentionne `"hachage"` 3 fois parmi 3000 mots peut obtenir un score comparable à un court document spécialisé qui le mentionne 3 fois parmi 300 mots, alors que le second est clairement plus focalisé. Il faut donc élaborer une formule plus complexe qu’un simple ratio sur la longueur.

## 5.2 La formule BM25

**BM25** (*Best Match 25*, 25e version de la famille de modèles *BM*) corrige ces deux limites :

$$\text{BM25}(t, d) = IDF(t) \times \frac{tf_{raw}(t,d) \cdot (k_1 + 1)}{tf_{raw}(t,d) + k_1 \cdot \left(1 - b + b \cdot \dfrac{|d|}{avgdl}\right)}$$

Où :
- $tf_{raw}(t, d)$ = nombre brut d'occurrences de `t` dans `d` (non normalisé)
- $|d|$ = longueur du document `d` en tokens
- $avgdl$ = longueur moyenne des documents dans le corpus
- $k_1 \in [1.2, 2.0]$ : paramètre de saturation du TF (valeur par défaut : **1.5**)
- $b \in [0, 1]$ : paramètre de normalisation par la longueur (valeur par défaut : **0.75**)

Le score total d'un document pour une requête est, comme pour TF-IDF, la somme sur les termes de la requête :

$$\text{score}_{BM25}(q, d) = \sum_{t \in q} \text{BM25}(t, d)$$

## 5.3 Interprétation des paramètres

On constate que la formule est nettement plus compliquée et difficile à appréhender. Vos camarades y reviendront dans la présentation de leurs mémoires, mais essayons déjà de construire une première intuition des paramètres de cette formule.

### Saturation via k₁

>Rappel : la saturation est la capacité à plafonner la contribution de la fréquence d’apparition des termes recherchés dans un document car il y a toujours un moment où le nombre de citation d’un terme donné n’augmente plus la pertinence

Observons le terme de droite de la fraction pour $b = 0$ (sans normalisation longueur) :

$$\frac{tf_{raw} \cdot (k_1 + 1)}{tf_{raw} + k_1}$$

Quand $tf_{raw} \to \infty$, ce terme converge vers $k_1 + 1$. Le score ne peut donc jamais dépasser $IDF(t) \times (k_1 + 1)$, quelle que soit la fréquence du terme dans le document. C'est bien une **saturation**.

- si $k_1$ petit (proche de 0) → saturation immédiate, la fréquence n'a presque aucun effet
- si $k_1$ grand → saturation lente, BM25 se rapproche de TF-IDF
- si $k_1 = 1.5$ → à partir de ~3-4 occurrences, les occurrences supplémentaires ont un effet marginal

```
Illustration de la saturation (k1 = 1.5) :

tf_raw =  1  → contribution BM25 ∝ 1 × 2.5 / (1 + 1.5)  = 1.00
tf_raw =  2  → contribution BM25 ∝ 2 × 2.5 / (2 + 1.5)  = 1.43 (+30%)
tf_raw =  5  → contribution BM25 ∝ 5 × 2.5 / (5 + 1.5)  = 1.92 (+26%)
tf_raw = 10  → contribution BM25 ∝ 10× 2.5 / (10 + 1.5) = 2.17 (+12%)
tf_raw = 50  → contribution BM25 ∝ 50× 2.5 / (50 + 1.5) = 2.42 (+10%)
tf_raw = ∞   → contribution BM25 ∝ k1 + 1               = 2.50 (+3%)  ← plafond

TF-IDF (référence) :
tf_raw =  1  → contribution TF-IDF ∝ 1
tf_raw = 10  → contribution TF-IDF ∝ 10    (×10)
tf_raw = 50  → contribution TF-IDF ∝ 50    (×50, illimité)
```

### Normalisation par la longueur via b

Le terme $\left(1 - b + b \cdot \frac{|d|}{avgdl}\right)$ dans le dénominateur ajuste la contribution selon la longueur du document :

- Si `b = 0` : le dénominateur vaut $tf_{raw} + k_1$, indépendant de la longueur → aucune pénalisation des longs documents
- Si `b = 1` : normalisation complète par la longueur, un document 2× plus long que la moyenne voit sa contribution divisée par ~2
- `b = 0.75` : compromis empirique standard. Il reconnaît qu'un document plus long a *aussi* plus de contenu, et ne doit pas être pénalisé aussi sévèrement qu'un document répétitif

## 5.4 BM25 dans l’industrie

BM25 est le moteur de scoring par défaut d'**Elasticsearch**, d'**Apache Lucene** (qui sous-tend de nombreux moteurs de recherche), et de **PostgreSQL Full-Text Search** depuis plusieurs années. Il donne de meilleurs résultats que TF-IDF sur des corpus hétérogènes (documents de longueurs très variées, textes avec répétitions intentionnelles) et ses deux paramètres $k_1$ et $b$ sont bien compris et ajustables selon le domaine.

En pratique, les valeurs $k_1 = 1.2$ et $b = 0.75$ (ou $k_1 = 1.5$ et $b = 0.75$) fonctionnent bien sur la plupart des corpus sans ajustement.

## 5.5 Architecture BM25

On part de la formule BM25 vue en section 5.2 et on se demande ce qu'il faut pour la calculer.

### Qu’est-ce qu'on veut calculer ?

La structure va être identique à TF-IDF. On va créer une méthode pour obtenir chaque « ingrédient » de la formule :
* un score par terme
* un score global par document
* et une méthode pour la recherche.

L'IDF qu’on va utiliser est identique à celui de TF-IDF, donc idf(term) est la même méthode. La nouveauté est le calcul du TF saturé et normalisé par la longueur, qui est suffisamment complexe pour mériter sa propre méthode `score_term(term, doc_id)`. Ensuite comme la classe précédente c’est une méthode spécifique, `score(query, doc_id)` qui va sommer les `score_term` sur les termes de la requête. Enfin `search(query, top_k)` reprend exactement la même logique que TFIDF.search.

### Qu’est-ce qu’il faut pour calculer `score_term` ? 

En regardant la formule, on y trouve : 
* `tf_raw` (le décompte brut de l’occurence des termes, comme dans TF-IDF)
* `k1`
* `b`

Les deux derniers sont les deux paramètres introduits et expliqués ci-dessus dans la section 5.2. On a également besoin de :
* `doc_lengths[doc_id]` (la longueur du document courant)
* avgdl (la longueur moyenne du corpus)

Par rapport à TF-IDF, la seule différence est `avgdl`. TF-IDF n'en avait pas besoin car il normalisait `term` par `term` dans `tf()`, tandis que BM25 normalise directement dans la formule de scoring en rapportant la longueur du document à la moyenne du corpus.

### Build
Faudra-t-il créer un `build` différent de ce qu’on a fait avec `TFIDF.build` ? Non : les deux ont besoin de `tf_raw` et `doc_lengths`. BM25 a juste besoin en plus de `avgdl`, qui se calcule en divisant la somme des `doc_lengths` par `n_docs`, une ligne supplémentaire à la fin du build, rien de plus.

### Comment  k1 et b sont-ils obtenus ?

On a vu plus haut qu’on avait besoin de `k1` et `b`, mais comment les obtient-on ?

Ce sont des attributs de la classe, passés à la construction. Cela permet de créer facilement deux instances avec des paramètres différents pour comparer leur comportement sur le même corpus, sans modifier le code de scoring.

### BM25 pourrait-elle être construit par héritage de TFIDF ?

Si plusieurs méthodes sont proches ou reprises de TFIDF (`search`, `buidl`), et des attributs communs (`tf_raw`, `doc_legnths`, `n_docs`…) on peut légitimement se demander si on ne pas créer BM25 par héritage.

C’est là qu’intervient le principe un peu abstrait de **substitution de Liskov** (LSP). Voilà ici un bon exemple pour le rendre plus concret.

Ce principe de substitution dit qu’une classe fille doit pouvoir remplacer sa classe mère partout sans changer le comportement attendu. Ici on a un problème. `TFIDF` possède une méthode `tf()`, or `BM25.tf()` n'existe pas car  BM25 n'a pas de TF normalisé séparé qui n’a pas vraiment de sens pour lui, il calcule directement `score_term`. Une classe BM25 héritant de TFIDF hériterait d'une méthode `tf()` qu'elle n'utilise pas, ce qui est trompeur.

La bonne approche serait plutôt de définir une classe abstraite `ScoringModel` (avec le module `abc.ABC` en Python) exposant les méthodes communes : `build`, `idf`, `score` et `search`, et de faire hériter TFIDF et BM25 séparément de cette classe abstraite. *C'est le principe de ségrégation des interfaces (ISP)* : on ne force pas BM25 à implémenter `tf()` dont il n'a pas besoin. Les deux classes partagent alors `build` et `search` par héritage depuis `ScoringModel`, et chacune implémente son propre calcul de score.

### Conception de la classe BM25
```
Classe BM25
   Attributs
      index       : InvertedIndex
      k1          : float   ← paramètre de saturation (défaut : 1.5)
      b           : float   ← paramètre de longueur (défaut : 0.75)
      tf_raw      : Dict[int, Dict[str, int]]  ← {doc_id → {terme → count brut}}
      doc_lengths : Dict[int, int]             ← longueur de chaque doc en tokens
      avgdl       : float                      ← longueur moyenne du corpus
      n_docs      : int

   Méthodes
      build(corpus) → None
         Pré-calcule tf_raw, doc_lengths, avgdl en une passe
         Identique à TFIDF.build — seule différence : on stocke tf_raw et non tf_norm

      idf(term) → float
         Identique à TFIDF.idf
         = log(n_docs / index.df(term))

      score_term(term, doc_id) → float
         numerator   = tf_raw[doc_id][term] * (k1 + 1)
         denominator = tf_raw[doc_id][term] + k1 * (1 - b + b * doc_lengths[doc_id] / avgdl)
         retourner idf(term) * numerator / denominator

      score(query, doc_id) → float
         = somme des score_term(t, doc_id) pour t dans tokenize(query)

      search(query, top_k) → List[Tuple[int, float]]
         Même logique que TFIDF.search : candidats via l'index, scoring, tri
```

> **Note d'implémentation :** l'IDF de BM25 varie légèrement selon les implémentations. La formule d'Elasticsearch ajoute un terme régularisateur pour éviter les scores négatifs sur les termes très fréquents : $IDF = \log\left(1 + \frac{N - df_t + 0.5}{df_t + 0.5}\right)$. Les deux formules donnent des classements très similaires en pratique.

# PARTIE 6 : N-grams et recherche approximative

## 6.1 Le problème des fautes de frappe

L'index inversé et TF-IDF/BM25 reposent sur des **tokens exacts**. Si l'utilisateur tape `"hachge"` (faute de frappe), le moteur ne trouve rien : `"hachge"` n'est pas dans l'index.

Plusieurs approches existent pour la tolérance aux fautes :
- **Distance de Levenshtein** (séance 5 ou alors vos propres recherches pour le mémoire) : mesure précise, mais O(m·n) par paire. Sur un vocabulaire de 100 000 termes, ce serait bien trop coûteux pour comparer chaque requête à tous les termes
- **N-grams de caractères** : pré-indexer des sous-séquences de caractères, permettant un filtrage rapide des candidats avant un re-classement fin

## 6.2 Définition des n-grams

Un **n-gram** est une sous-séquence contiguë de `n` caractères extraite d'une chaîne. Les n-grams de caractères (à ne pas confondre avec les n-grams de mots) permettent de comparer des chaînes par leur structure locale.

Convention : on entoure la chaîne de marqueurs `$` (début et fin) pour distinguer les positions en bordure de mot.

Pour `"hachage"` avec `n = 3` (trigrammes) :

```
Chaîne padded : "$hachage$"

Trigrammes : { "$ha", "hac", "ach", "cha", "hag", "age", "ge$" }
```

Une faute de frappe ne modifie qu'un petit nombre de trigrammes :

```
"hachage" → { "$ha", "hac", "ach", "cha", "hag", "age", "ge$" }   (7 trigrammes)
"hachge"  → { "$ha", "hac", "ach", "chg", "hge", "ge$" }           (6 trigrammes)

Communs : { "$ha", "hac", "ach", "ge$" }  → 4 sur 9 uniques au total
```

Les deux chaînes partagent la majorité de leurs trigrammes malgré la faute. C'est l'intuition exploitée par la similarité de Jaccard.

## 6.3 Similarité de Jaccard

La **similarité de Jaccard** entre deux ensembles $A$ et $B$ est :

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- Si $A = B$ → $J = 1$ (identique)
- Si $A \cap B = \emptyset$ → $J = 0$ (aucun trigramme en commun)
- Une faute de frappe ne modifie que quelques trigrammes → $J$ reste élevé

```
Exemples (n=3) :

J("hachage", "hachage")  = 1.000   ← identique
J("hachage", "hachge")   = 0.444   ← transposition, reste élevé
J("hachage", "hachages") = 0.625   ← pluriel, très proche
J("hachage", "hashage")  = 0.333   ← erreur au milieu, plus distant
J("hachage", "collision")= 0.000   ← aucun trigramme en commun
```

## 6.4 Index n-gram pour la recherche approximative

La recherche approximative par Jaccard sur des n-grams ne se fait pas terme à terme (trop coûteux), mais via un **index sur les n-grams** :

```
{ trigramme → liste de termes du vocabulaire qui le contiennent }

"$ha" : ["hachage", "hacher", "haricot", ...]
"hac" : ["hachage", "hache", "hachoir", ...]
"ach" : ["hachage", "acheter", "achat", ...]
...
```

Requête approximative pour `"hachge"` :
1. Extraire les trigrammes de `"hachge"` : `{"$ha", "hac", "ach", "chg", "hge", "ge$"}`
2. Pour chaque trigramme, récupérer la liste de candidats depuis l'index n-gram
3. Compter le nombre de trigrammes en commun pour chaque candidat (candidats les plus fréquents = les plus similaires)
4. Calculer $J$ pour les meilleurs candidats et ne retenir que ceux au-dessus d'un seuil

Cette approche évite de comparer la requête à chaque terme du vocabulaire : seuls les termes partageant au moins un trigramme avec la requête sont considérés.

## 6.5 N-grams vs Levenshtein : quand utiliser quoi ?

| | N-grams (Jaccard) | Distance de Levenshtein |
|---|---|---|
| **Complexité** | O(L) pour l'extraction, O(|candidats|) pour le scoring | O(m·n) par paire |
| **Passage à l'échelle** | Excellente (index n-gram) | Mauvaise sur grand vocabulaire |
| **Précision** | Bonne mais sensible à la taille de n | Précise, compte exactement les opérations |
| **Usage type** | Pré-filtrage rapide | Re-classement fin des candidats |

En pratique, les deux se **combinent** : l'index n-gram produit une liste restreinte de candidats, Levenshtein les re-classe. C'est exactement le pipeline de PostgreSQL `pg_trgm` : filtrage par trigrammes via l'index, puis score de similarité précis.

## 6.6 Architecture de l'index n-gram

Voici une proposition pour créer une classe `NgramIndex`, vous pouvez à titre d’exercice (ou dans les mini-mémoire) poser une réflexion pour concevoir une telle classe

```
Classe NgramIndex
   Attributs
      n              : int                       ← taille des n-grams (défaut : 3)
      index          : Dict[str, Set[str]]       ← {trigramme → ensemble de termes}
      vocabulary     : List[str]                 ← tous les termes indexés
      ngrams_cache   : Dict[str, Set[str]]       ← cache {terme → ses n-grams}

   Méthodes
      ngrams(word: str) → Set[str]
         Génère les n-grams de word avec marqueurs $
         Résultat mis en cache

      build(vocabulary: List[str]) → None
         Pour chaque terme du vocabulaire :
            pour chaque n-gram du terme :
               ajouter terme à index[n-gram]

      candidates(query_word: str, min_common: int = 1) → List[str]
         Extrait les n-grams de query_word
         Rassemble tous les termes présents dans au moins min_common de ces n-grams
         Retourne la liste de candidats

      jaccard(word_a: str, word_b: str) → float
         = |ngrams(word_a) ∩ ngrams(word_b)| / |ngrams(word_a) ∪ ngrams(word_b)|

      fuzzy_search(query_word: str, threshold: float = 0.3) → List[Tuple[str, float]]
         Récupère les candidats via l'index n-gram
         Calcule J pour chaque candidat
         Retourne les candidats avec J ≥ threshold, triés par J décroissant
```

# PARTIE 7 : Pour aller plus loin

## Synthèse : architecture d'un moteur de recherche textuelle

Les cinq parties de ce notebook constituent les briques d'un moteur de recherche complet. Voici comment elles s'articulent :

```
INGESTION (une seule fois)
   Corpus brut
      → Preprocessor.process()         → tokens normalisés
      → InvertedIndex.build()           → posting lists triées
      → TFIDF.build() / BM25.build()   → tf_raw, doc_lengths, avgdl
      → NgramIndex.build()             → index trigrammes sur le vocabulaire

REQUÊTE (à chaque fois)
   Requête utilisateur
      → Preprocessor.process()          → tokens de la requête
      → [optionnel] NgramIndex.fuzzy_search() → correction des fautes
      → InvertedIndex.search_and()      → doc_ids candidats
      → BM25.score() pour chaque candidat → scores
      → tri par score décroissant        → top-k résultats
```
## Récapitulatif : choisir la bonne approche

| Besoin | Approche recommandée |
|---|---|
| Filtrage exact multi-termes rapide | Index inversé + recherche AND |
| Classement par pertinence, corpus homogène | TF-IDF |
| Classement par pertinence, corpus hétérogène (longueurs variées) | BM25 |
| Tolérance aux fautes de frappe | Index n-gram + Jaccard |
| Compréhension sémantique (synonymes, paraphrases) | Embeddings (séance 7) |
| Système complet en production | BM25 + n-grams + embeddings pour re-ranking |

## Lien avec les mini-mémoires

* **Sujet 8 : Moteur de recherche textuelle (TF-IDF, BM25, index inversé)**. Implémentation complète des classes décrites ci-dessus *from scratch*. Vous pouvez utiliser sklearn.TfidfVectorizer (ou une autre bibliothèque) pour vérifier / benchmarker les résultats. Approfondissez cette notion d’évaluation avec La **précision@k** qui mesure la fraction de documents pertinents parmi les `k` premiers résultats. Regardez la notion de *ground truth*. Techniques : annoter manuellement 20-30 requêtes, ou construire un corpus où la pertinence est encodée par construction (documents générés par catégories, requêtes ciblant une catégorie). Renseignez-vous aussi sur les méthodes de compression des *postings lists* quand le corpus est immense (*delta encoding* par exemple, juste exposer le principe). Regarder l’usage de `PostgreSQL pg_trgm`.

* **Sujet 9 : Bases vectorielles et recherche sémantique**. Le présent cours est un prérequis direct pour la séance 7 et pour ce sujet. TF-IDF produit un vecteur creux (*sparse*) dans un espace de dimension égale à la taille du vocabulaire (souvent 50 000+). Un embeddin* (BERT, sentence-transformers) est un vecteur dense de dimension fixe (768) appris par un réseau de neurones, qui encode non plus la fréquence des mots mais leur sémantique. Les limites de BM25 que vous aurez identifiées dans le sujet 8 (sensibilité aux synonymes, incapacité à gérer les paraphrases) sont exactement ce que les embeddings corrigent. Le sujet 9 vous demandera de construire un système hybride : BM25 pour un premier filtrage rapide (top-50), embeddings pour un re-ranking sémantique fin (top-10). Ce pipeline est le standard industriel actuel dans les RAG (*Retrieval-Augmented Generation*).

## Références

- **Manning, Raghavan & Schütze**, *Introduction to Information Retrieval* (Cambridge, 2008). La référence académique du domaine, [disponible gratuitement en ligne](https://nlp.stanford.edu/IR-book/pdf/irbookonlinereading.pdf). Voir Chapitres 1 (index inversé), 6 (scoring), 11 (TF-IDF), 11.4 (BM25).
- **Robertson & Zaragoza** (2009) — *The Probabilistic Relevance Framework: BM25 and Beyond*. [L'article des auteurs de BM25 expliquant les fondements probabilistes et les extensions](https://www.researchgate.net/publication/220613776_The_Probabilistic_Relevance_Framework_BM25_and_Beyond).
- **Documentation Elasticsearch** (*Similarity module*)
- **Documentation PostgreSQL** (*pg_trgm*)
- **Lucene documentation** (*BM25Similarity*)